In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from datasets import load_dataset
import random

ds = load_dataset("tatsu-lab/alpaca")
sft_list = list(ds["train"])
random.shuffle(sft_list)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [ ]:
def format_sample(row):
  if row["input"]:
    prompt = f"Instrcution: {row["instruction"]}\nInput: {row["input"]}\nResponse:\n"
  else:
    prompt = f"Instruction: {row["instruction"]}\nResponse:\n"
  return prompt, row["output"]

In [ ]:
model_name = "facebook/opt-125m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)

# check trainable params
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
trainable = sum(p.numel() for p in model.parameters())

print(f"Total: {total}")
print(f"Trainable (Before Lora): {trainable} ({100*trainable/total:.2f}%)")

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/251M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Total: 163848192
Trainable (Before Lora): 163848192 (100.00%)


In [ ]:
def tokenize_sample(row, max_length=512):
  prompt, response = format_sample(row)
  full = prompt + response
  full_ids = tokenizer(full)["input_ids"]
  prompt_ids = tokenizer(prompt)["input_ids"]
  prompt_len = len(prompt_ids)

  full_ids = full_ids[:max_length]
  x = torch.tensor(full_ids)
  y = x.clone()
  y[:prompt_len] = -100

  if (y!=100).sum() == 0:
    return None
  return x, y

tokenized = [tokenize_sample(row) for row in sft_list]
tokenized = [t for t in tokenized if t is not None]
print(f"valid samples: {len(tokenized)}")

model.safetensors:   0%|          | 0.00/251M [00:00<?, ?B/s]

valid samples: 52002


In [ ]:
import torch.nn.functional as F

def get_batch(data, batch_size=8):
  samples = random.sample(data, batch_size)
  xs, ys = zip(*samples)
  max_len = max(x.shape[0] for x in xs)
  xs = torch.stack([F.pad(x, (0, max_len-x.size(0)), value=tokenizer.pad_token_id) for x in xs])
  ys = torch.stack([F.pad(y, (0, max_len-y.size(0)), value=-100) for y in ys])
  return xs.to(device), ys.to(device)

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, linear, rank=8, alpha=16):
        super().__init__()

        self.linear = linear              # original frozen weight
        self.rank   = rank
        self.alpha  = alpha

        d_out, d_in = linear.weight.shape

        # trainable low rank matrices
        self.A = nn.Parameter(torch.randn(rank, d_in) * 0.01)  # small random init
        self.B = nn.Parameter(torch.zeros(d_out, rank))         # zero init → LoRA starts as identity

        # freeze original weight
        self.linear.weight.requires_grad = False
        if self.linear.bias is not None:
            self.linear.bias.requires_grad = False

    def forward(self, x):
        base = self.linear(x)                           # frozen path: W @ x
        lora = x @ self.A.T @ self.B.T                 # trainable path: x @ A^T @ B^T
        return base + (self.alpha / self.rank) * lora   # combine

In [ ]:
# nn.Module   # container
              # has forward method
              # can contain Parameters and other Modules
              # e.g. nn.Linear, nn.Embedding, your whole model

def apply_lora(model, rank=8, alpha=16, target_modules=["q_proj", "v_proj"]): # only Q and V
  # W_q (frozen) + lora = Q
  for name, module in model.named_modules(): #
    for target in target_modules:
      if name.endswith(target) and isinstance(module, nn.Linear):
        # get parent module
        parent_name = ".".join(name.split(".")[:-1]) # str
        child_name = name.split(".")[-1] # str => linear.bias vs linear."bias" => therefore use setattr for linear."bias"
        parent = model.get_submodule(parent_name) # parent: module object

        # replace with LoRA version
        setattr(parent, child_name, LoRALinear(module, rank, alpha)) # LoRALinear is a module
        print(f"replaced {name} with LoRA")

  return model

In [ ]:
# freeze parameters
for param in model.parameters():
    param.requires_grad = False

# apply lora
lora_model = apply_lora(model, rank=8, alpha=16)

# check trainable params
total = sum(p.numel() for p in lora_model.parameters())
trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)

print(f"Total: {total}")
print(f"Trainable (After Lora): {trainable} ({100*trainable/total:.2f}%)")

replaced model.decoder.layers.0.self_attn.v_proj with LoRA
replaced model.decoder.layers.0.self_attn.q_proj with LoRA
replaced model.decoder.layers.1.self_attn.v_proj with LoRA
replaced model.decoder.layers.1.self_attn.q_proj with LoRA
replaced model.decoder.layers.2.self_attn.v_proj with LoRA
replaced model.decoder.layers.2.self_attn.q_proj with LoRA
replaced model.decoder.layers.3.self_attn.v_proj with LoRA
replaced model.decoder.layers.3.self_attn.q_proj with LoRA
replaced model.decoder.layers.4.self_attn.v_proj with LoRA
replaced model.decoder.layers.4.self_attn.q_proj with LoRA
replaced model.decoder.layers.5.self_attn.v_proj with LoRA
replaced model.decoder.layers.5.self_attn.q_proj with LoRA
replaced model.decoder.layers.6.self_attn.v_proj with LoRA
replaced model.decoder.layers.6.self_attn.q_proj with LoRA
replaced model.decoder.layers.7.self_attn.v_proj with LoRA
replaced model.decoder.layers.7.self_attn.q_proj with LoRA
replaced model.decoder.layers.8.self_attn.v_proj with Lo

In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, lora_model.parameters()),
    lr=1e-4 # keeps params where requires_grad is True
)

lora_model = lora_model.to(device)

for step in range(2000):
  x, y = get_batch(tokenized)
  x, y = x.to(device), y.to(device)
  outputs = lora_model(input_ids=x, labels=y) # X: (B, T); Y: (B, T)
  loss = outputs.loss

  optimizer.zero_grad()
  loss.backward()
  torch.nn.utils.clip_grad_norm_(lora_model.parameters(), 1.0)
  optimizer.step()

  if step % 100 == 0:
    print(f"step {step} | loss {loss.item():.4f}")


step 0 | loss 2.5219
step 100 | loss 2.0310
step 200 | loss 2.3644
step 300 | loss 2.2176
step 400 | loss 2.3408
step 500 | loss 2.4359
step 600 | loss 2.1174
step 700 | loss 2.4272
step 800 | loss 2.3341
step 900 | loss 2.3576
step 1000 | loss 2.2121
step 1100 | loss 2.0512
step 1200 | loss 1.8155
step 1300 | loss 2.3105
step 1400 | loss 1.9052
step 1500 | loss 2.4172
step 1600 | loss 2.6605
step 1700 | loss 2.4895
step 1800 | loss 2.4810
step 1900 | loss 2.0558


In [ ]:
# save full model (with lora weights baked in)
torch.save(lora_model.state_dict(), "/content/drive/MyDrive/opt_lora.pt")

# or save only lora weights (tiny — ~2MB)
lora_state = {k: v for k, v in lora_model.state_dict().items() if ".A" in k or ".B" in k}
torch.save(lora_state, "/content/drive/MyDrive/opt_lora_weights_only.pt")

print(f"full model saved: {sum(v.numel() for v in lora_model.state_dict().values()):,} params")
print(f"lora only saved:  {sum(v.numel() for v in lora_state.values()):,} params")

full model saved: 164,143,104 params
lora only saved:  294,912 params


In [ ]:
lora_state   = torch.load("/content/drive/MyDrive/opt_lora_weights_only.pt", map_location=device)
current_state = model.state_dict()

# merge lora weights into current state
current_state.update(lora_state)
model.load_state_dict(current_state)

model.eval()
print("loaded!")

loaded!


In [ ]:
# generate
def generate(prompt, max_new_tokens=200):
    inputs    = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = 0.7,
            do_sample          = True,
            repetition_penalty = 1.3,
            pad_token_id       = tokenizer.eos_token_id
        )
    response = out[0][prompt_len:]
    return tokenizer.decode(response, skip_special_tokens=True)

# test
print(generate("Instruction: What are 3 benefits of exercise?\nResponse:\n"))

3. It is beneficial for health and overall wellness. 
4. Its value to improve the quality of life, mental well-being, mood control, or other physical functions in a healthy way can be substantial.
5. Exercise improves circulation. This helps reduce blood sugar levels and decreases inflammation which may result from high body fat. 
6. It offers an opportunity to explore new areas of fitness such as hiking/running/jogging, dancing/blinding/sliding/dancing etc.. 
7. Athletes receive an appreciation for their efforts by becoming more active while at work. 
8. The use of weights allows them to focus on different aspects of their lives without being distracted themselves. 
9. And finally, it provides a chance for others to have fun doing things that you might not otherwise enjoy (such as sports). 
10. People get some help with managing stress due to the fact they do so much for each day. 

